In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ayush2222/structured-bgl-logs-csv/BGL.log_structured.csv


In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/ayush2222/structured-bgl-logs-csv/BGL.log_structured.csv


In [4]:
import pandas as pd

chunks = []
chunk_size = 20000   # small safe size

for chunk in pd.read_csv(
    "/kaggle/input/datasets/ayush2222/structured-bgl-logs-csv/BGL.log_structured.csv",
    chunksize=chunk_size
):
    chunks.append(chunk)
    
    # stop after 5 chunks (≈100k rows)
    if len(chunks) == 5:
        break

df = pd.concat(chunks)

print("Loaded shape:", df.shape)
df.head()

Loaded shape: (100000, 14)


,LineId,Label,Timestamp,Date,Node,Time,NodeRepeat,Type,Component,Level,Content,EventId,EventTemplate,ParameterList
0,1,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.363779,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
1,2,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.527847,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
2,3,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
3,4,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.823719,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]
4,5,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.982731,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,3aa50e45,instruction cache parity error corrected,[]


In [5]:
df.to_csv("/kaggle/working/BGL_small.csv", index=False)

In [6]:
import pandas as pd

def extract_features(df):
    
    # Convert timestamp
    df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')
    
    # Group logs by Node (like a machine/user)
    grouped = df.groupby('Node')

    features = []

    for node, group in grouped:
        
        total_logs = len(group)
        
        # Count severity levels
        info_count = (group['Level'] == 'INFO').sum()
        fatal_count = (group['Level'] == 'FATAL').sum()
        
        # Unique events
        unique_events = group['EventId'].nunique()
        
        # Time behavior
        time_span = group['Timestamp'].max() - group['Timestamp'].min()
        
        features.append({
            "Node": node,
            "total_logs": total_logs,
            "info_count": info_count,
            "fatal_count": fatal_count,
            "unique_events": unique_events,
            "time_span": time_span
        })

    return pd.DataFrame(features)

In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/working/BGL_small.csv")

print("Loaded")
print(df.shape)
print(df.head())

Loaded
(100000, 14)
   LineId Label   Timestamp        Date                 Node  \
0       1     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
1       2     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
2       3     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
3       4     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
4       5     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   

                         Time           NodeRepeat Type Component Level  \
0  2005-06-03-15.42.50.363779  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
1  2005-06-03-15.42.50.527847  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
2  2005-06-03-15.42.50.675872  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
3  2005-06-03-15.42.50.823719  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
4  2005-06-03-15.42.50.982731  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   

                                    Content   EventId  \
0  instruction cache parity error corrected  3aa50e45   
1  instruction cache parity er

In [8]:
print("Columns:", df.columns)

Columns: Index(['LineId', 'Label', 'Timestamp', 'Date', 'Node', 'Time', 'NodeRepeat',
       'Type', 'Component', 'Level', 'Content', 'EventId', 'EventTemplate',
       'ParameterList'],
      dtype='object')


In [9]:
print("Unique nodes:", df['Node'].nunique())

Unique nodes: 24065


In [10]:
grouped = df.groupby('Node')

for node, group in grouped:
    print("Node:", node)
    print("Logs:", len(group))
    break

Node: R00-M0-N0-C:J03-U01
Logs: 1


In [11]:
def extract_features(df):
    print("Function started")
    
    df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')
    
    grouped = df.groupby('Node')

    features = []

    for i, (node, group) in enumerate(grouped):
        
        if i < 5:   # only print first 5
            print("Processing:", node)
        
        total_logs = len(group)

        features.append({
            "Node": node,
            "total_logs": total_logs
        })

    print("Loop finished")

    return pd.DataFrame(features)

In [12]:
features = extract_features(df)

print(features.head())
print("Shape:", features.shape)

Function started
Processing: R00-M0-N0-C:J03-U01
Processing: R00-M0-N0-C:J04-U01
Processing: R00-M0-N0-C:J05-U11
Processing: R00-M0-N0-C:J06-U11
Processing: R00-M0-N0-C:J07-U01
Loop finished
                  Node  total_logs
0  R00-M0-N0-C:J03-U01           1
1  R00-M0-N0-C:J04-U01           1
2  R00-M0-N0-C:J05-U11           1
3  R00-M0-N0-C:J06-U11           1
4  R00-M0-N0-C:J07-U01           1
Shape: (24065, 2)


In [13]:
def extract_features(df):
    
    df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')

    # Create time window (VERY IMPORTANT)
    df['time_window'] = df['Timestamp'] // 5000

    grouped = df.groupby('time_window')

    features = []

    for window, group in grouped:
        
        total_logs = len(group)
        
        info_count = (group['Level'] == 'INFO').sum()
        fatal_count = (group['Level'] == 'FATAL').sum()
        
        unique_nodes = group['Node'].nunique()
        unique_events = group['EventId'].nunique()

        features.append({
            "window": window,
            "total_logs": total_logs,
            "info_count": info_count,
            "fatal_count": fatal_count,
            "unique_nodes": unique_nodes,
            "unique_events": unique_events
        })

    return pd.DataFrame(features)

In [14]:
features = extract_features(df)

print(features.head())
print("Shape:", features.shape)

   window  total_logs  info_count  fatal_count  unique_nodes  unique_events
0  223567        4896        4896            0            29              6
1  223568        4988        4948           40          4683             10
2  223569        4776        4776            0          4254              9
3  223570          11          11            0             7              6
4  223572          35          35            0             8              5
Shape: (24, 6)


In [15]:
#Creating labels
def create_labels(features):
    
    features['target'] = features['fatal_count'].apply(lambda x: 1 if x > 0 else 0)
    
    return features

data = create_labels(features)

print(data)

    window  total_logs  info_count  fatal_count  unique_nodes  unique_events  \
0   223567        4896        4896            0            29              6   
1   223568        4988        4948           40          4683             10   
2   223569        4776        4776            0          4254              9   
3   223570          11          11            0             7              6   
4   223572          35          35            0             8              5   
5   223573        1053          29         1024           520              9   
6   223577        1046          22         1024           521              6   
7   223578          41          41            0             7              4   
8   223579           1           1            0             1              1   
9   223581           2           2            0             2              1   
10  223582           1           1            0             1              1   
11  223583           1           1      

In [16]:
print(data['target'].value_counts())

target
0    14
1    10
Name: count, dtype: int64


In [17]:
#Training the model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = data[['total_logs', 'info_count', 'fatal_count', 'unique_nodes', 'unique_events']]
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



In [18]:
#Detect Alerts
data['prediction'] = model.predict(X)
alerts = data[data['prediction'] == 1]
print("🚨 ALERTS DETECTED:")
print(alerts)

🚨 ALERTS DETECTED:
    window  total_logs  info_count  fatal_count  unique_nodes  unique_events  \
1   223568        4988        4948           40          4683             10   
5   223573        1053          29         1024           520              9   
6   223577        1046          22         1024           521              6   
17  223590         154          26          128            70              6   
18  223591       16425       16170          255          4104              8   
19  223592         179         133           46            31             42   
20  223594       12153       11467          686         10944             34   
21  223595       16971       16667          304         13490             36   
22  223596       12093        9436         2657          5652             36   
23  223597       24713       24615           98         10142              6   

    target  prediction  
1        1           1  
5        1           1  
6        1           1  


In [21]:
#using isolation forest, it detects anomalies without labels Unsupervised
from sklearn.ensemble import IsolationForest

X = features[['total_logs', 'info_count', 'fatal_count', 'unique_nodes', 'unique_events']]

model = IsolationForest(contamination=0.2)  # 20% anomalies assumption
model.fit(X)

features['anomaly'] = model.predict(X)

# Convert (-1 → anomaly, 1 → normal)
features['anomaly'] = features['anomaly'].apply(lambda x: 1 if x == -1 else 0)

alerts = features[features['anomaly'] == 1]

print("🚨 UNSUPERVISED ALERTS:")
print(alerts)

🚨 UNSUPERVISED ALERTS:
    window  total_logs  info_count  fatal_count  unique_nodes  unique_events  \
18  223591       16425       16170          255          4104              8   
20  223594       12153       11467          686         10944             34   
21  223595       16971       16667          304         13490             36   
22  223596       12093        9436         2657          5652             36   
23  223597       24713       24615           98         10142              6   

    target  prediction  anomaly_score  anomaly  
18       1           1       0.015525        1  
20       1           1       0.056447        1  
21       1           1       0.017913        1  
22       1           1       0.219714        1  
23       1           1       0.003966        1  


In [22]:
#We're adding some anomalies score
features['anomaly_score'] = model.decision_function(X)
print(features.sort_values(by='anomaly_score').head())

    window  total_logs  info_count  fatal_count  unique_nodes  unique_events  \
23  223597       24713       24615           98         10142              6   
22  223596       12093        9436         2657          5652             36   
21  223595       16971       16667          304         13490             36   
20  223594       12153       11467          686         10944             34   
18  223591       16425       16170          255          4104              8   

    target  prediction  anomaly_score  anomaly  
23       1           1      -0.116260        1  
22       1           1      -0.111320        1  
21       1           1      -0.065374        1  
20       1           1      -0.041901        1  
18       1           1      -0.020865        1  
